In [0]:
# ── Fill these in from your kaggle.json ──────────────────────────
KAGGLE_USERNAME = "user_name"
KAGGLE_KEY      = "kaggle_api_key"
# ─────────────────────────────────────────────────────────────────

import os, json

# Step 1: install kaggle package
%pip install -q kaggle

# Step 2: write credentials to driver home (avoids DBFS I/O error)
kaggle_dir = os.path.expanduser("~/.config/kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)

os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)
print("Kaggle credentials set ✓")

In [0]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

# Download to driver local disk (avoids DBFS Fuse errors)
download_path = "/tmp/retail_raw"
os.makedirs(download_path, exist_ok=True)

# Dataset: carrie1/ecommerce-data  (~45MB, real UK retail orders)
api.dataset_download_files(
    "carrie1/ecommerce-data",
    path=download_path,
    unzip=True
)

# See what was downloaded
files = os.listdir(download_path)
print(f"Downloaded files: {files}")

In [0]:
import pandas as pd

# Read the CSV from local driver disk
csv_path = f"{download_path}/data.csv"

df_pd = pd.read_csv(csv_path, encoding="ISO-8859-1")
print(f"Raw shape: {df_pd.shape}")
print(df_pd.head(3))

# Convert to Spark DataFrame and save as Delta table
spark_df = spark.createDataFrame(df_pd)

spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.raw_retail_source")

count = spark.table("default.raw_retail_source").count()
print(f"\nSaved to Delta table. Total rows: {count:,}")
spark.table("default.raw_retail_source").show(5, truncate=False)